In [1]:
import requests
from tqdm import notebook
from bs4 import BeautifulSoup

In [2]:
author_page_url = 'https://m-strana.ru/blog/authors/?PAGEN_1='

In [3]:
AUTHOR_LINKS = []
for i in range(2, 9):
    author_page_url = f'{author_page_url}{i}'
    author_page_r = requests.get(author_page_url)
    if author_page_r.status_code == 200:
        author_page_soup = BeautifulSoup(author_page_r.content, features='lxml')
        author_block = author_page_soup.find('ul', {'class': 'authors-list pagination-item'})
        for li in author_block.find_all('li'):
            link = li.find('a', {'class': 'name'})
            if link:
                AUTHOR_LINKS.append(f"https://m-strana.ru{link['href']}")

In [4]:
ARTICLE_LINKS = []
for link in notebook.tqdm(AUTHOR_LINKS):
    author_r = requests.get(link)
    if author_r.status_code == 200:
        author_s = BeautifulSoup(author_r.content)
        art_section = author_s.find('ul', {'class': 'posts-list extended pagination-item'})
        ARTICLE_LINKS.extend([f"https://m-strana.ru{link['href']}" for link in art_section.find_all('a')])

  0%|          | 0/140 [00:00<?, ?it/s]

In [2]:
from threading import Thread
from tqdm import notebook

In [6]:
def download_page(links, storage):
    for link in notebook.tqdm(links):
        art_r = requests.get(link)
        if art_r.status_code == 200:
            art_s = BeautifulSoup(art_r.content)
            art_text = art_s.find('article', {'class': 'article-content'}).text
            storage.append(art_text)
            

In [7]:
n_threads = 30
STORAGE = [list() for i in range(n_threads)]
threads = []
for i in range(n_threads):
    cur_links = ARTICLE_LINKS[i*len(ARTICLE_LINKS)//n_threads:(i+1)*len(ARTICLE_LINKS)//n_threads+2]
    thread = Thread(target=download_page, args=(cur_links, STORAGE[i]))
    thread.start()
    threads.append(thread)
for thread in threads:
    thread.join()

  0%|          | 0/91 [00:00<?, ?it/s]

  0%|          | 0/91 [00:00<?, ?it/s]

  0%|          | 0/92 [00:00<?, ?it/s]

  0%|          | 0/91 [00:00<?, ?it/s]

  0%|          | 0/92 [00:00<?, ?it/s]

...

  0%|          | 0/92 [00:00<?, ?it/s]

  0%|          | 0/92 [00:00<?, ?it/s]

  0%|          | 0/91 [00:00<?, ?it/s]

  0%|          | 0/90 [00:00<?, ?it/s]

Exception in thread Thread-27 (download_page):
Traceback (most recent call last):
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 488, in _make_request
    raise new_e
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 1093, in _validate_conn
    conn.connect()
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connection.py", line 741, in connect
    sock_and_verified = _ssl_wrap_socket_and_match_hostname(
  File "C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\urllib3\connection.py", line 920, in _ssl_wrap_socket_and_match_hostna

In [8]:
CORPUS = [text for storage in STORAGE for text in storage]

In [9]:
CORPUS = [t.strip() for t in CORPUS]

In [10]:
for i, doc in enumerate(CORPUS):
    with open(f'corpus_raw/{i}.txt', 'w', encoding='utf-8') as f:
        f.write(doc)

In [3]:
import os

In [4]:
CORPUS = []
for f in os.listdir('corpus_raw'):
    with open(f'corpus_raw/{f}', 'r', encoding='utf-8') as f:
        CORPUS.append('\n'.join([line.strip() for line in f.readlines() if line.strip()]))

In [5]:
import spacy
nlp = spacy.load('ru_core_news_sm')

In [6]:
def annotate_files(docs, storage):
    for doc in notebook.tqdm(docs):
        try:
            text = ''
            for sent in nlp(doc).sents:
                sent_text = ''
                for tk in sent:
                    if tk.text.isalpha():
                        sent_text += f'{tk.lemma_.lower()}_{tk.pos_} '
                text += f'{sent_text.strip()}\n'
            storage.append(text)
        except Exception as e:
            continue

In [7]:
n_threads = 6
ANNOTATED_CORPUS = [list() for i in range(n_threads)]
threads = []
for i in range(n_threads):
    cur_articles = CORPUS[i*len(CORPUS)//n_threads:(i+1)*len(CORPUS)//n_threads+2]
    thread = Thread(target=annotate_files, args=(cur_articles, ANNOTATED_CORPUS[i]))
    thread.start()
    threads.append(thread)
for thread in threads:
    thread.join()

  0%|          | 0/458 [00:00<?, ?it/s]

  0%|          | 0/459 [00:00<?, ?it/s]

  0%|          | 0/459 [00:00<?, ?it/s]

  0%|          | 0/458 [00:00<?, ?it/s]

  0%|          | 0/458 [00:00<?, ?it/s]

  0%|          | 0/457 [00:00<?, ?it/s]

In [8]:
ANNOTATED_CORPUS_FULL = [text for storage in ANNOTATED_CORPUS for text in storage]

In [9]:
for i, doc in enumerate(ANNOTATED_CORPUS_FULL):
    with open(f'annotated_corpus/{i}.txt', 'w', encoding='utf-8') as f:
        f.write(doc)

In [10]:
tokens = [text.split() for text in ANNOTATED_CORPUS_FULL]
tokens = [tk for text in tokens for tk in text]

In [11]:
len(tokens), len(set(tokens))

(2681847, 29155)